# Aula 3 — Companion prático

## RAG com LangChain e Chroma

Este notebook agora funciona como **companion leve** para o material em `src/genai/rag/`.

Objetivos:

1. visualizar a arquitetura de RAG como pipeline;
2. inspecionar os módulos separados;
3. observar chunks, retrieval e geração sem reimplementar tudo no notebook;
4. manter o notebook como ambiente de experimento, não como fonte principal da implementação.

Implementação canônica:

- [README de RAG](/Users/ebezerra/ailab/gcc1734/src/genai/rag/README.md)
- [pipeline_demo.py](/Users/ebezerra/ailab/gcc1734/src/genai/rag/pipeline_demo.py)


## 0. Instalação

Execute a próxima célula apenas se o ambiente ainda não tiver as dependências.

O notebook depende dos mesmos pacotes usados pelos módulos em `src/genai/rag/`.


In [ ]:
# Execute apenas se precisar instalar as dependências.
# %pip install -q langchain langchain-community langchain-text-splitters langchain-openai langchain-ollama python-dotenv chromadb


## 1. Imports

Nesta versão, o notebook importa os módulos do diretório `src/genai/rag/` em vez de definir o pipeline inteiro aqui dentro.


In [ ]:
from src.genai.rag.sample_corpus import ensure_sample_document
from src.genai.rag.loaders import load_text_documents
from src.genai.rag.chunking import split_documents
from src.genai.rag.indexing import (
    build_embedding_model,
    build_vectorstore,
    default_persist_directory,
)
from src.genai.rag.retrieval import build_retriever, retrieve_context
from src.genai.rag.generation import build_rag_prompt, generate_answer
from src.genai.rag.llm_config import load_env, get_llm


## 2. Visão modular do pipeline

O pipeline de RAG foi separado em camadas:

1. `sample_corpus.py`: cria ou aponta para o documento de exemplo;
2. `loaders.py`: carrega documentos;
3. `chunking.py`: divide o texto em pedaços menores;
4. `indexing.py`: gera embeddings e cria o índice vetorial;
5. `retrieval.py`: recupera chunks relevantes;
6. `generation.py`: injeta contexto no prompt e chama o LLM.

Essa estrutura deixa mais clara a diferença entre:

- indexação;
- recuperação;
- geração.


In [ ]:
env_path = load_env()
sample_path = ensure_sample_document()
docs = load_text_documents(sample_path)

print(".env carregado de:", env_path or "nenhum arquivo encontrado")
print("Documento de exemplo:", sample_path)
print("Documentos carregados:", len(docs))
print()
print(docs[0].page_content)


## 3. Chunking

Agora usamos diretamente o módulo `chunking.py`.

O objetivo aqui é inspecionar a saída do splitter, não reimplementar a lógica.


In [ ]:
splitter, chunks = split_documents(docs, chunk_size=80, chunk_overlap=20)

print("Splitter:", type(splitter).__name__)
print("Número de chunks:", len(chunks))
print()
for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk.page_content}")


## 4. Embeddings e indexação

Esta etapa usa `indexing.py` para:

1. criar o modelo de embeddings;
2. criar o Chroma local;
3. persistir o índice em disco.

Observação:

esta célula requer credenciais válidas se você usar `OpenAIEmbeddings`.


In [ ]:
embedding_model = build_embedding_model()
vectorstore = build_vectorstore(
    chunks,
    persist_directory=default_persist_directory(),
    embedding_model=embedding_model,
)

print("Persist directory:", default_persist_directory())
print("Vectorstore:", type(vectorstore).__name__)


## 5. Retrieval

Com o índice criado, usamos `retrieval.py` para montar o retriever e recuperar os chunks mais relevantes para uma pergunta.


In [ ]:
retriever = build_retriever(vectorstore, k=2)

question = "What is artificial intelligence?"
retrieved_docs, context = retrieve_context(retriever, question)

print("Question:", question)
print()
print("Retrieved chunks:")
for doc in retrieved_docs:
    print("-", doc.page_content)

print()
print("Context passed to generation:")
print(context)


## 6. Geração com contexto recuperado

Agora usamos `generation.py` para construir o prompt de RAG e produzir a resposta final.

Observação:

esta célula chama um LLM real.


In [ ]:
prompt = build_rag_prompt()
llm = get_llm("openai")  # troque para "ollama" se preferir

answer = generate_answer(llm, question, context, prompt)

print("ANSWER:")
print(answer)


## 7. Função auxiliar para testar várias perguntas

O notebook continua útil como ambiente de experimentação.

Mas agora a lógica principal continua nos scripts de `src/genai/rag/`.


In [ ]:
def rag_demo(question: str, k: int = 2, provider: str = "openai") -> None:
    local_retriever = build_retriever(vectorstore, k=k)
    docs_found, local_context = retrieve_context(local_retriever, question)
    local_llm = get_llm(provider)
    local_prompt = build_rag_prompt()
    local_answer = generate_answer(local_llm, question, local_context, local_prompt)

    print("=" * 60)
    print("QUESTION:")
    print(question)
    print("-" * 60)
    print("RETRIEVED:")
    for doc in docs_found:
        print("-", doc.page_content)
    print("-" * 60)
    print("ANSWER:")
    print(local_answer)
    print("=" * 60)


In [ ]:
# Descomente para testar com mais perguntas.
# rag_demo("What is a database?")
# rag_demo("What are LLM-based agents?")
# rag_demo("What is Quantum Physics?")


## 8. O que este notebook mostra agora

Este notebook deixou de ser a implementação principal do RAG.

Ele agora serve para:

1. inspecionar a saída de cada etapa;
2. experimentar perguntas diferentes;
3. observar como os módulos separados interagem;
4. manter a implementação reutilizável fora do `.ipynb`.

Se a ideia for estudar a arquitetura, leia:

- [sample_corpus.py](/Users/ebezerra/ailab/gcc1734/src/genai/rag/sample_corpus.py)
- [chunking.py](/Users/ebezerra/ailab/gcc1734/src/genai/rag/chunking.py)
- [indexing.py](/Users/ebezerra/ailab/gcc1734/src/genai/rag/indexing.py)
- [retrieval.py](/Users/ebezerra/ailab/gcc1734/src/genai/rag/retrieval.py)
- [generation.py](/Users/ebezerra/ailab/gcc1734/src/genai/rag/generation.py)
- [pipeline_demo.py](/Users/ebezerra/ailab/gcc1734/src/genai/rag/pipeline_demo.py)


## 9. Exercícios

1. Altere `chunk_size` e `chunk_overlap` e compare os chunks gerados.
2. Troque o corpus de exemplo por um arquivo maior.
3. Mude `k` no retriever e observe o efeito na resposta.
4. Compare `openai` e `ollama` como provider.
5. Rode o script [pipeline_demo.py](/Users/ebezerra/ailab/gcc1734/src/genai/rag/pipeline_demo.py) no terminal e compare a experiência com a do notebook.
